In [ ]:
# data

import tensorflow as tf
from tensorflow import keras as k
from tensorflow.keras import layers as l
import numpy as np
import pandas as pd

tf.random.set_seed(42)
np.random.seed(42)

(x,y),(x_test,y_test)=k.datasets.cifar10.load_data()

y=k.utils.to_categorical(y,10)
y_test=k.utils.to_categorical(y_test,10)

x_train,x_val=x[:40000],x[40000:]
y_train,y_val=y[:40000],y[40000:]

def preprocess(img,label):
  img=tf.cast(img,tf.float32)
  img=tf.image.resize(img,(160,160))

  # preprocess aqui
  img=k.applications.efficientnet.preprocess_input(img)

  return img,label

train_ds=tf.data.Dataset.from_tensor_slices((x_train,y_train))\
  .shuffle(10000)\
  .map(preprocess)\
  .batch(64)\
  .prefetch(tf.data.AUTOTUNE)

val_ds=tf.data.Dataset.from_tensor_slices((x_val,y_val))\
  .map(preprocess)\
  .batch(64)\
  .prefetch(tf.data.AUTOTUNE)

test_ds=tf.data.Dataset.from_tensor_slices((x_test,y_test))\
  .map(preprocess)\
  .batch(64)\
  .prefetch(tf.data.AUTOTUNE)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [ ]:
# experiment 1

base=k.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

base.trainable=False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
data_aug=k.Sequential([
    l.RandomRotation(0.1),
    l.RandomZoom(0.1)
])

model=k.Sequential([
    l.Input(shape=(160,160,3)),
    data_aug,
    base,

    l.GlobalAveragePooling2D(),
    l.Dense(128,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.3),

    # float32 importante
    l.Dense(10,activation='softmax',dtype='float32')
])

model.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
625/625 - 110s - 176ms/step - accuracy: 0.7723 - loss: 0.6859 - val_accuracy: 0.8915 - val_loss: 0.3082
Epoch 2/10
625/625 - 55s - 89ms/step - accuracy: 0.8212 - loss: 0.5215 - val_accuracy: 0.8961 - val_loss: 0.2956
Epoch 3/10
625/625 - 54s - 87ms/step - accuracy: 0.8352 - loss: 0.4833 - val_accuracy: 0.9004 - val_loss: 0.2863
Epoch 4/10
625/625 - 55s - 88ms/step - accuracy: 0.8404 - loss: 0.4656 - val_accuracy: 0.9043 - val_loss: 0.2724
Epoch 5/10
625/625 - 55s - 88ms/step - accuracy: 0.8478 - loss: 0.4423 - val_accuracy: 0.9033 - val_loss: 0.2779
Epoch 6/10
625/625 - 55s - 87ms/step - accuracy: 0.8468 - loss: 0.4411 - val_accuracy: 0.9071 - val_loss: 0.2691
Epoch 7/10
625/625 - 55s - 88ms/step - accuracy: 0.8532 - loss: 0.4240 - val_accuracy: 0.9050 - val_loss: 0.2714
Epoch 8/10
625/625 - 55s - 87ms/step - accuracy: 0.8559 - loss: 0.4161 - val_accuracy: 0.9076 - val_loss: 0.2693
Epoch 9/10
625/625 - 55s - 88ms/step - accuracy: 0.8583 - loss: 0.4101 - val_accuracy: 0.9073 

In [ ]:
base.trainable=True

for layer in base.layers[:-30]:
  layer.trainable=False

model.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ckpt_30=k.callbacks.ModelCheckpoint(
    'model_30_layers.keras',
    monitor='val_loss',
    save_best_only=True
)

cb=[
    k.callbacks.EarlyStopping(patience=3,restore_best_weights=True),
    k.callbacks.ReduceLROnPlateau(patience=2,factor=0.3,min_lr=1e-6),
    ckpt_30
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    verbose=2,
    callbacks=cb
)

model.evaluate(test_ds)

# final save
model.save('model_30_layers_final.keras')

Epoch 1/20
625/625 - 93s - 149ms/step - accuracy: 0.8017 - loss: 0.5851 - val_accuracy: 0.8877 - val_loss: 0.3390 - learning_rate: 1.0000e-05
Epoch 2/20
625/625 - 65s - 104ms/step - accuracy: 0.8329 - loss: 0.4867 - val_accuracy: 0.8968 - val_loss: 0.3061 - learning_rate: 1.0000e-05
Epoch 3/20
625/625 - 65s - 104ms/step - accuracy: 0.8436 - loss: 0.4528 - val_accuracy: 0.9031 - val_loss: 0.2865 - learning_rate: 1.0000e-05
Epoch 4/20
625/625 - 65s - 104ms/step - accuracy: 0.8517 - loss: 0.4299 - val_accuracy: 0.9065 - val_loss: 0.2729 - learning_rate: 1.0000e-05
Epoch 5/20
625/625 - 65s - 104ms/step - accuracy: 0.8565 - loss: 0.4126 - val_accuracy: 0.9111 - val_loss: 0.2631 - learning_rate: 1.0000e-05
Epoch 6/20
625/625 - 65s - 104ms/step - accuracy: 0.8649 - loss: 0.3926 - val_accuracy: 0.9145 - val_loss: 0.2553 - learning_rate: 1.0000e-05
Epoch 7/20
625/625 - 67s - 108ms/step - accuracy: 0.8676 - loss: 0.3820 - val_accuracy: 0.9172 - val_loss: 0.2480 - learning_rate: 1.0000e-05
Epoch 

In [ ]:
# experimet 2

base_2=k.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

base_2.trainable=False

In [ ]:
data_aug2=k.Sequential([
    l.RandomRotation(0.1),
    l.RandomZoom(0.1)
])

model_2=k.Sequential([
    l.Input(shape=(160,160,3)),
    data_aug2,
    base_2,

    l.GlobalAveragePooling2D(),
    l.Dense(128,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.3),

    l.Dense(10,activation='softmax',dtype='float32')
])

model_2.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
625/625 - 70s - 112ms/step - accuracy: 0.7708 - loss: 0.6924 - val_accuracy: 0.8915 - val_loss: 0.3187
Epoch 2/10
625/625 - 55s - 88ms/step - accuracy: 0.8251 - loss: 0.5158 - val_accuracy: 0.8997 - val_loss: 0.2912
Epoch 3/10
625/625 - 82s - 132ms/step - accuracy: 0.8358 - loss: 0.4829 - val_accuracy: 0.9008 - val_loss: 0.2844
Epoch 4/10
625/625 - 54s - 87ms/step - accuracy: 0.8409 - loss: 0.4651 - val_accuracy: 0.9064 - val_loss: 0.2757
Epoch 5/10
625/625 - 55s - 88ms/step - accuracy: 0.8454 - loss: 0.4512 - val_accuracy: 0.9060 - val_loss: 0.2706
Epoch 6/10
625/625 - 54s - 87ms/step - accuracy: 0.8506 - loss: 0.4301 - val_accuracy: 0.9081 - val_loss: 0.2721
Epoch 7/10
625/625 - 55s - 88ms/step - accuracy: 0.8528 - loss: 0.4268 - val_accuracy: 0.9045 - val_loss: 0.2734
Epoch 8/10
625/625 - 55s - 87ms/step - accuracy: 0.8550 - loss: 0.4140 - val_accuracy: 0.9051 - val_loss: 0.2669
Epoch 9/10
625/625 - 55s - 88ms/step - accuracy: 0.8564 - loss: 0.4144 - val_accuracy: 0.9113 

In [ ]:
base_2.trainable=True

for layer in base_2.layers[:-40]:
  layer.trainable=False

model_2.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ckpt_40=k.callbacks.ModelCheckpoint(
    'model_40_layers.keras',
    monitor='val_loss',
    save_best_only=True
)

cb2=[
    k.callbacks.EarlyStopping(patience=3,restore_best_weights=True),
    k.callbacks.ReduceLROnPlateau(patience=2,factor=0.3,min_lr=1e-6),
    ckpt_40
]

model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    verbose=2,
    callbacks=cb2
)

model_2.evaluate(test_ds)

# final save
model_2.save('model_40_layers_final.keras')

Epoch 1/20
625/625 - 86s - 138ms/step - accuracy: 0.8062 - loss: 0.5750 - val_accuracy: 0.8915 - val_loss: 0.3251 - learning_rate: 1.0000e-05
Epoch 2/20
625/625 - 66s - 106ms/step - accuracy: 0.8382 - loss: 0.4705 - val_accuracy: 0.9020 - val_loss: 0.2898 - learning_rate: 1.0000e-05
Epoch 3/20
625/625 - 66s - 106ms/step - accuracy: 0.8534 - loss: 0.4275 - val_accuracy: 0.9091 - val_loss: 0.2697 - learning_rate: 1.0000e-05
Epoch 4/20
625/625 - 66s - 106ms/step - accuracy: 0.8612 - loss: 0.4077 - val_accuracy: 0.9122 - val_loss: 0.2561 - learning_rate: 1.0000e-05
Epoch 5/20
625/625 - 66s - 106ms/step - accuracy: 0.8690 - loss: 0.3775 - val_accuracy: 0.9152 - val_loss: 0.2455 - learning_rate: 1.0000e-05
Epoch 6/20
625/625 - 66s - 106ms/step - accuracy: 0.8752 - loss: 0.3661 - val_accuracy: 0.9188 - val_loss: 0.2379 - learning_rate: 1.0000e-05
Epoch 7/20
625/625 - 82s - 132ms/step - accuracy: 0.8761 - loss: 0.3576 - val_accuracy: 0.9214 - val_loss: 0.2294 - learning_rate: 1.0000e-05
Epoch 

In [ ]:
# experiment 3

base_3=k.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

base_3.trainable=False

In [ ]:
data_aug3=k.Sequential([
    l.RandomRotation(0.1),
    l.RandomZoom(0.1)
])

model_3=k.Sequential([
    l.Input(shape=(160,160,3)),
    data_aug3,
    base_3,

    l.GlobalAveragePooling2D(),
    l.Dense(128,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.3),

    l.Dense(10,activation='softmax',dtype='float32')
])

model_3.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
625/625 - 69s - 110ms/step - accuracy: 0.7711 - loss: 0.6805 - val_accuracy: 0.8906 - val_loss: 0.3176
Epoch 2/10
625/625 - 80s - 127ms/step - accuracy: 0.8173 - loss: 0.5281 - val_accuracy: 0.8965 - val_loss: 0.3001
Epoch 3/10
625/625 - 55s - 87ms/step - accuracy: 0.8327 - loss: 0.4852 - val_accuracy: 0.8966 - val_loss: 0.2853
Epoch 4/10
625/625 - 55s - 88ms/step - accuracy: 0.8404 - loss: 0.4641 - val_accuracy: 0.9011 - val_loss: 0.2818
Epoch 5/10
625/625 - 82s - 131ms/step - accuracy: 0.8489 - loss: 0.4464 - val_accuracy: 0.9067 - val_loss: 0.2697
Epoch 6/10
625/625 - 54s - 87ms/step - accuracy: 0.8470 - loss: 0.4465 - val_accuracy: 0.9086 - val_loss: 0.2622
Epoch 7/10
625/625 - 55s - 87ms/step - accuracy: 0.8540 - loss: 0.4221 - val_accuracy: 0.9081 - val_loss: 0.2675
Epoch 8/10
625/625 - 55s - 87ms/step - accuracy: 0.8569 - loss: 0.4191 - val_accuracy: 0.9083 - val_loss: 0.2649
Epoch 9/10
625/625 - 55s - 88ms/step - accuracy: 0.8573 - loss: 0.4120 - val_accuracy: 0.9107

In [ ]:
base_3.trainable=True

for layer in base_3.layers[:-50]:
  layer.trainable=False

model_3.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ckpt_50=k.callbacks.ModelCheckpoint(
    'model_50_layers.keras',
    monitor='val_loss',
    save_best_only=True
)

cb3=[
    k.callbacks.EarlyStopping(patience=3,restore_best_weights=True),
    k.callbacks.ReduceLROnPlateau(patience=2,factor=0.3,min_lr=1e-6),
    ckpt_50
]

model_3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    verbose=2,
    callbacks=cb3
)

model_3.evaluate(test_ds)

# final save
model_3.save('model_50_layers_final.keras')

Epoch 1/20
625/625 - 92s - 148ms/step - accuracy: 0.7935 - loss: 0.6174 - val_accuracy: 0.8882 - val_loss: 0.3369 - learning_rate: 1.0000e-05
Epoch 2/20
625/625 - 70s - 112ms/step - accuracy: 0.8334 - loss: 0.4906 - val_accuracy: 0.9010 - val_loss: 0.2949 - learning_rate: 1.0000e-05
Epoch 3/20
625/625 - 70s - 112ms/step - accuracy: 0.8475 - loss: 0.4437 - val_accuracy: 0.9073 - val_loss: 0.2735 - learning_rate: 1.0000e-05
Epoch 4/20
625/625 - 70s - 113ms/step - accuracy: 0.8576 - loss: 0.4200 - val_accuracy: 0.9117 - val_loss: 0.2577 - learning_rate: 1.0000e-05
Epoch 5/20
625/625 - 71s - 113ms/step - accuracy: 0.8646 - loss: 0.3908 - val_accuracy: 0.9168 - val_loss: 0.2450 - learning_rate: 1.0000e-05
Epoch 6/20
625/625 - 81s - 130ms/step - accuracy: 0.8709 - loss: 0.3775 - val_accuracy: 0.9197 - val_loss: 0.2368 - learning_rate: 1.0000e-05
Epoch 7/20
625/625 - 70s - 112ms/step - accuracy: 0.8777 - loss: 0.3582 - val_accuracy: 0.9232 - val_loss: 0.2296 - learning_rate: 1.0000e-05
Epoch 

In [ ]:
# final dataframe

results=[]

for name,model_x in [
    ('30 layers',model),
    ('40 layers',model_2),
    ('50 layers',model_3),
]:

    loss,acc=model_x.evaluate(
        test_ds,
        verbose=0
    )

    results.append([
        name,
        acc,
        loss
    ])

df=pd.DataFrame(
    results,
    columns=[
        'experiment',
        'test_accuracy',
        'test_loss'
    ]
)

print(df)

df.to_csv(
    'results.csv',
    index=False
)

  experiment  test_accuracy  test_loss
0  30 layers         0.9286   0.214152
1  40 layers         0.9339   0.197051
2  50 layers         0.9350   0.194034


In [ ]:
# download colab

from google.colab import files

files.download('model_30_layers_final.keras')
files.download('model_40_layers_final.keras')
files.download('model_50_layers_final.keras')

files.download('results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>